# S6E10 — RealMLP (diversity model)

Companion to `airline_satisfaction_prediction.ipynb`. After round 7 the four tree models there agree at Spearman
0.982–0.995, and the remaining gap to the strongest public notebook is model diversity rather than features (see
its section 6). This notebook trains a neural network, RealMLP, on the same rows and writes its out-of-fold and test
predictions to `../preds/`. The main notebook picks them up, and its blend decides whether they stay.

- **Folds:** the main notebook's 5-fold ablation split (`StratifiedKFold(5, shuffle=True, random_state=42)`), so the
  OOF predictions line up row for row. It is also the split goodpjw2008 uses, so the scores are comparable.
- **Recipe:** the public RealMLP recipe as used in goodpjw2008's S6E10 notebooks (hyperparameters credited there to
  Vladimir Demidov): the raw columns plus a categorical twin of every numeric column, so the network learns one
  embedding per distinct value, with the `flat_anneal` learning-rate schedule. In their measurements this
  raw-input variant scored about 0.9609 alone, yet added more to their stack than stronger, more tree-like
  variants.
- **A separate kernel:** torch stays out of the GBDT notebook. On the machine this was built on, torch and the GBDT
  libraries' OpenMP runtimes can deadlock in one process, so torch runs single-threaded here and the network
  trains on the Apple GPU (MPS).

Two input variants, chosen with `INPUTS`:

- `'raw'` (rounds 8–9): the raw columns plus the categorical twins.
- `'enc'` (round 10): the same, plus the main notebook's engineered features: the original-data model's
  prediction, the per-value rates from the original survey, the value counts, and the fold-safe target
  encodings. The main notebook exports them (section 4 there); goodpjw2008's version of this variant was
  their best single model.

Run with `SMOKE = True` first. One epoch on the full fold-0 training set checks that the run fits in memory and
times an epoch at full scale, so the length of the full run is known before committing to it.

In [1]:
import torch
torch.set_num_threads(1)     # first, before other libraries load their own OpenMP runtimes

import gc
import hashlib
import os
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from scipy.stats import rankdata
from pytabkit import RealMLP_TD_Classifier
from IPython.display import display

import logging
import warnings
warnings.filterwarnings('ignore')
for name in ('lightning', 'lightning.pytorch', 'lightning.fabric', 'pytorch_lightning'):
    logging.getLogger(name).setLevel(logging.ERROR)      # keep the per-fit training log quiet

SEED = 42            # fold split: must match ABL_FOLDS in the main notebook
INPUTS = 'enc'       # 'raw': raw columns + categorical twins (rounds 8-9); 'enc': + the main notebook's features
MODEL_SEED = 42      # network initialisation and batch order (raw runs: 42 in round 8, 7 in round 9)
TARGET = 'satisfaction'
DATA_DIR = '../data'
PRED_DIR = '../preds'
os.makedirs(PRED_DIR, exist_ok=True)

# True: one epoch on the full fold-0 training set, to check memory and time an epoch at full scale.
# Nothing is saved. (A first smoke run on a 60k-row subset passed, but the full run then ran out of GPU memory.)
# Full-scale smoke run with n_ens=4 (raw inputs): 50s per epoch, fold-0 AUC 0.9604 after one epoch, 3.3 GB.
# Full-scale smoke run with the enc inputs: 56s per epoch, fold-0 AUC 0.9608 after one epoch, 3.3 GB.
SMOKE = False
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', DEVICE, '| inputs:', INPUTS, '| model seed:', MODEL_SEED, '| smoke run:', SMOKE)

device: mps | inputs: enc | model seed: 42 | smoke run: False


## 1. Data and folds

In [2]:
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
train_df[TARGET] = train_df[TARGET].astype(str).str.strip().str.lower().eq('true').astype('int8')
y = train_df[TARGET].values

CAT_COLS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUM_COLS = [c for c in test_df.columns if c not in CAT_COLS + ['id']]     # 4 continuous + 13 ratings

# Identical to ABL_FOLDS in the main notebook: same labels, same arguments, so the OOF rows line up
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(train_df, y))
print('train', train_df.shape, '| test', test_df.shape, '| folds', [len(va) for _, va in FOLDS])

train (699635, 23) | test (299844, 22) | folds [139927, 139927, 139927, 139927, 139927]


## 2. Inputs

RealMLP does its own preprocessing (robust scaling, smooth clipping, one-hot or embeddings), so the inputs are the
raw columns with two changes:

- **A categorical twin of every numeric column.** `Flight Distance` has about 3,500 distinct values; as a category
  it gets one learned embedding per value, which is the network's version of the exact-value encodings that
  carried rounds 3 and 4 of the main notebook. The numeric original stays, so the ordering is not lost.
- **The 0.04% missing `Arrival Delay` values are set to 0**, the median, because RealMLP needs numeric inputs to be
  complete.

The categories are built on train and test together, so both share one set of levels.

With `INPUTS = 'enc'` the main notebook's features join these columns, read from the files its section 4
exports. Two of them are transformed for the network: the original-data model's probability becomes a logit (it
sits close to 0 or 1), and the value counts are log-transformed. Values the original survey or a training fold
never saw have no encoding; they get the column mean, or the base rate for the target encodings.

In [3]:
def nn_frame(df):
    """Raw columns plus a categorical twin of every numeric column."""
    X = df[CAT_COLS + NUM_COLS].copy()
    X['Arrival Delay in Minutes'] = X['Arrival Delay in Minutes'].fillna(0.0)
    cat_cols = list(CAT_COLS)
    for c in NUM_COLS:
        X[f'{c}_cat'] = X[c].astype(int).astype(str)
        cat_cols.append(f'{c}_cat')
    for c in cat_cols:
        X[c] = X[c].astype(str).astype('category')
    return X, cat_cols


X_all, NN_CATS = nn_frame(pd.concat([train_df, test_df], ignore_index=True))
X = X_all.iloc[:len(train_df)].reset_index(drop=True)
X_test = X_all.iloc[len(train_df):].reset_index(drop=True)
NN_CARD = max((X_all[c].cat.categories.size, c) for c in NN_CATS)
del X_all, train_df, test_df       # GPU and CPU share 16 GB of memory here; keep only what training needs
gc.collect()
print(X.shape, X_test.shape, '|', len(NN_CATS), 'categorical columns |',
      'largest:', NN_CARD)

# --- the main notebook's features (INPUTS = 'enc') ---
FEAT_DIR = f'{PRED_DIR}/features_orig_pred+orig_te+freq+te+te_pair_k5'
if INPUTS == 'enc':
    assert os.path.exists(f'{FEAT_DIR}/done'), 'run the main notebook first: its section 4 exports these features'
    z = np.load(f'{FEAT_DIR}/static.npz')
    ST_TRAIN = pd.DataFrame(z['train'], columns=list(z['cols']))
    ST_TEST = pd.DataFrame(z['test'], columns=list(z['cols']))
    for df in (ST_TRAIN, ST_TEST):
        p = df['orig_pred'].clip(1e-6, 1 - 1e-6)
        df['orig_pred'] = np.log(p / (1 - p))                  # probabilities near 0 / 1 -> logit
        for c in [c for c in df.columns if c.startswith('freq_')]:
            df[c] = np.log1p(df[c])
    col_means = ST_TRAIN.mean()                                 # values the original never saw
    ST_TRAIN, ST_TEST = ST_TRAIN.fillna(col_means), ST_TEST.fillna(col_means)
    PRIOR = y.mean()
    print('engineered features:', ST_TRAIN.shape[1], 'static +',
          len(np.load(f'{FEAT_DIR}/te_fold0.npz', allow_pickle=True)['cols']), 'target encodings per fold')


def fold_inputs(k, tr, va):
    """(training, validation, test) inputs for fold k."""
    if INPUTS == 'raw':
        return X.iloc[tr], X.iloc[va], X_test
    # allow_pickle: the first export stored these column names as an object array; the files are our own
    z = np.load(f'{FEAT_DIR}/te_fold{k}.npz', allow_pickle=True)
    te_train = pd.DataFrame(np.nan_to_num(z['train'], nan=PRIOR), columns=list(z['cols']))
    te_test = pd.DataFrame(np.nan_to_num(z['test'], nan=PRIOR), columns=list(z['cols']))
    full = pd.concat([X, ST_TRAIN, te_train], axis=1)
    return full.iloc[tr], full.iloc[va], pd.concat([X_test, ST_TEST, te_test], axis=1)

(699635, 38) (299844, 38) | 21 categorical columns | largest: (3521, 'Flight Distance_cat')
engineered features: 26 static + 7 target encodings per fold


## 3. Model

The validation fold is passed to `fit` so that RealMLP keeps its best epoch by validation AUC, the network's
counterpart of the early stopping the tree models use in the main notebook.

The recipe trains `n_ens=8` networks side by side; here it is 4. With 8, the full run ran out of memory on a
16 GB Mac, where the CPU and the GPU share the same memory (goodpjw2008 trained on a dedicated 16 GB GPU).
goodpjw2008 found that extra seeds of this model added nothing to its out-of-fold score, so averaging
four networks instead of eight should cost little.

pytabkit resets torch's thread count to the number of CPU cores inside `fit` and `predict` unless `n_threads` is
given, so `n_threads=1` is passed explicitly to keep the single-thread setting from the first cell.

Each run uses its own `MODEL_SEED` (network initialisation and batch order) while the fold split stays fixed,
and the main notebook averages the runs listed in its `EXTERNAL_PREDS`. Two runs of four networks amount to the
recipe's eight.

Each fold's predictions are saved as soon as it finishes. If the kernel dies, re-running this cell resumes at the
first missing fold.

In [4]:
# Public RealMLP recipe as used in goodpjw2008's S6E10 notebooks (hyperparameters credited there to
# Vladimir Demidov), with the flat_anneal schedule from the same series.
REALMLP = dict(
    n_ens=4,                  # recipe: n_ens=8, which ran out of memory here (see above)
    n_epochs=4, batch_size=256, use_early_stopping=False,
    lr=0.053, wd=0.0236, sq_mom=0.988, lr_sched='flat_anneal', wd_sched='cos_log_15',
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu', plr_lr_factor=0.1151, plr_sigma=2.33,
    ls_eps=0.01, ls_eps_sched='sqrt_cos', add_front_scale=False, bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'],
)
RUN = dict(device=DEVICE, n_threads=1, verbosity=0, val_metric_name='1-auc_ovr')


GPU_GB = {}     # memory the MPS driver held at the end of the last fit, to see the headroom


def fit_predict(k, tr_idx, va_idx, seed, test=True, **override):
    """Fit on fold k with its validation rows as the validation set; return validation and test predictions."""
    X_tr, X_va, X_te = fold_inputs(k, tr_idx, va_idx)
    m = RealMLP_TD_Classifier(**{**REALMLP, **override}, **RUN, random_state=seed)
    m.fit(X_tr, y[tr_idx], X_val=X_va, y_val=y[va_idx], cat_col_names=NN_CATS)
    p_eval = m.predict_proba(X_va)[:, 1]
    p_test = m.predict_proba(X_te)[:, 1] if test else None
    if DEVICE == 'mps':
        GPU_GB['last'] = torch.mps.driver_allocated_memory() / 1e9
    del m
    gc.collect()
    if DEVICE == 'mps':
        torch.mps.empty_cache()
    return p_eval, p_test


# Round 8's run (MODEL_SEED 42, file realmlp_raw_k5_f3b590) predates a fix to the line above, where batch_size and
# use_early_stopping had slipped into a comment; both equal pytabkit's defaults (256, False), so that run is the
# same model, but its hash was computed without them.
RUN_ID = hashlib.md5(repr((sorted(REALMLP.items()), MODEL_SEED)).encode()).hexdigest()[:6]
STEM = f'{PRED_DIR}/realmlp_{INPUTS}_k{len(FOLDS)}_{RUN_ID}'
print('predictions will be saved as', STEM + '_oof.npy / _test.npy')

predictions will be saved as ../preds/realmlp_enc_k5_fc8ab0_oof.npy / _test.npy


In [5]:
%%time
if SMOKE:
    # One epoch on the full fold-0 training set: the same memory footprint as the real run, at a quarter of its
    # time per fold. If this fits, the full run fits.
    tr, va = FOLDS[0]
    t0 = time.time()
    p, _ = fit_predict(0, tr, va, MODEL_SEED, test=False, n_epochs=1)
    secs = time.time() - t0
    print(f'smoke run: one epoch on {len(tr):,} rows in {secs:.0f}s | fold-0 AUC after one epoch '
          f'{roc_auc_score(y[va], p):.4f} | GPU memory held by the driver: {GPU_GB.get("last", float("nan")):.1f} GB')
    est_min = secs * REALMLP['n_epochs'] * len(FOLDS) / 60
    print(f'estimated full run: roughly {est_min:.0f} minutes, plus the test predictions')
else:
    oof, pred = np.zeros(len(X)), np.zeros(len(X_test))
    for fold, (tr, va) in enumerate(FOLDS):
        part = f'{STEM}_fold{fold}.npz'
        if os.path.exists(part):                       # resume after a crash
            z = np.load(part)
            oof[va], p_test = z['oof'], z['test']
            print(f'fold {fold}  loaded {part}')
        else:
            t0 = time.time()
            oof[va], p_test = fit_predict(fold, tr, va, MODEL_SEED)
            np.savez(part, oof=oof[va], test=p_test)
            print(f'fold {fold}  auc={roc_auc_score(y[va], oof[va]):.6f}  ({time.time() - t0:.0f}s, '
                  f'GPU {GPU_GB.get("last", float("nan")):.1f} GB)')
        pred += p_test / len(FOLDS)

    np.save(f'{STEM}_oof.npy', oof)
    np.save(f'{STEM}_test.npy', pred)
    for fold in range(len(FOLDS)):
        os.remove(f'{STEM}_fold{fold}.npz')
    print(f'\nrealmlp  OOF AUC = {roc_auc_score(y, oof):.6f} | log loss = {log_loss(y, oof):.6f}')

fold 0  auc=0.961129  (195s, GPU 3.3 GB)
fold 1  auc=0.959841  (199s, GPU 3.3 GB)
fold 2  auc=0.961729  (200s, GPU 3.3 GB)
fold 3  auc=0.961175  (201s, GPU 3.3 GB)
fold 4  auc=0.960927  (203s, GPU 3.3 GB)

realmlp  OOF AUC = 0.960956 | log loss = 0.222858
CPU times: user 9min 5s, sys: 2min 41s, total: 11min 46s
Wall time: 16min 38s


## 4. Does it add anything?

A quick read before the main notebook's blend: the round 9 blend is rebuilt from its saved predictions with its
recorded weights, and this run enters with a single weight scanned from 0 to 0.5. Its correlation with each member
shows whether it brings a new view or repeats one.

This is a screening estimate (the weight is chosen on the same OOF rows it is scored on). The decision belongs to
the main notebook's blend.

In [6]:
if not SMOKE:
    # Round 9 blend: members (file stems, averaged when several) and weights, as recorded in the main notebook
    ROUND9 = {
        'lgb': (['lgb_orig_pred+orig_te+freq+te+te_pair_k10_58dc21'], 0.110),
        'xgb': (['xgb_orig_pred+orig_te+freq+te+te_pair_k10_43abf3'], 0.121),
        'cat': (['cat_orig_pred+orig_te+freq+te+te_pair_k10_b649ff'], 0.124),
        'cat_allcat': (['cat_allcat_orig_pred+orig_te+freq+te+te_pair_k5_d4bef2'], 0.273),
        'realmlp_raw': (['realmlp_raw_k5_f3b590', 'realmlp_raw_k5_7da5ad'], 0.372),
    }
    rank = lambda v: rankdata(v) / len(v)
    members = {n: np.mean([np.load(f'{PRED_DIR}/{s}_oof.npy') for s in stems], axis=0)
               for n, (stems, _) in ROUND9.items()}
    base = sum(w * rank(members[n]) for n, (_, w) in ROUND9.items())

    corr = pd.DataFrame({**members, 'this run': oof}).corr(method='spearman')['this run'].drop('this run')
    print(f'this run alone: {roc_auc_score(y, oof):.6f}')
    print('Spearman correlation with the round 9 members:', corr.round(4).to_dict())

    scan = pd.Series({w: roc_auc_score(y, (1 - w) * base + w * rank(oof)) for w in np.round(np.arange(0, 0.51, 0.05), 2)})
    print(f'\nround 9 blend alone: {scan[0.0]:.6f}')
    print(f'best with this run : {scan.max():.6f} at weight {scan.idxmax()}  ({scan.max() - scan[0.0]:+.6f})')
    display(scan.round(6).to_frame('OOF AUC').T)

this run alone: 0.960956
Spearman correlation with the round 9 members: {'lgb': 0.9743, 'xgb': 0.974, 'cat': 0.9755, 'cat_allcat': 0.9776, 'realmlp_raw': 0.9844}

round 9 blend alone: 0.961496
best with this run : 0.961499 at weight 0.05  (+0.000003)


,0.00,0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0.50
OOF AUC,0.961496,0.961499,0.961499,0.961495,0.961489,0.961478,0.961465,0.961448,0.961428,0.961405,0.961379


**Result (round 8 of the main notebook).** OOF AUC 0.960883 with `n_ens=4`, close to the 0.96094–0.96098
goodpjw2008 reports for the same recipe with `n_ens=8`; about 3 minutes per fold on the M4's GPU, at 3.3 GB.

The point is the correlation. Against the four tree models it is 0.967–0.972, where the trees sit at 0.982–0.995
among themselves, so it is the first member that ranks passengers in a genuinely different way. In the screening
above it lifts the round 7 blend from 0.961290 to 0.961486 (+0.000196) at weight 0.35: the largest single step
since round 4. The main notebook loads these predictions as its fifth model.

**Round 10 result (`INPUTS = 'enc'`).** OOF AUC 0.960956, a little above the raw-input model (0.960883 for one seed,
0.960948 for two), as goodpjw2008 found. But the engineered features are the same ones the trees use, and the
network now ranks passengers much more like them: Spearman 0.974–0.978 against the trees, where the raw-input
RealMLP sits at 0.968–0.973, and 0.984 with the raw-input RealMLP itself. Added to the round 9 blend it is worth
+0.000003 at weight 0.05, which is noise. RealMLP's value here is its difference from the trees, and feeding it their
features removes it, so the main notebook leaves this run out.

## Credits

- **goodpjw2008**, *S6E10 | Two-View GBDT + RealMLP Stack* and the rest of that series: the RealMLP inputs (a
  categorical twin of every numeric column) and the `flat_anneal` schedule.
- **Vladimir Demidov**: the RealMLP hyperparameters, as credited in goodpjw2008's notebooks.
- **The pytabkit authors**: RealMLP itself.